In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/df_encoded.csv')
y = pd.read_csv('../data/processed/y.csv').squeeze()

print(f"Shape: {df.shape}")
print(f"Target: {y.shape}")

Shape: (2919, 237)
Target: (1460,)


In [2]:
# Cases granja/familiar — fora del con per baix
drop_idx = df.iloc[:1460][
    (df.iloc[:1460]['GrLivArea'] > 4000) & 
    (np.expm1(y) < 200000)
].index

print(df.iloc[drop_idx][['GrLivArea']].join(np.expm1(y.iloc[drop_idx]).rename('SalePrice')))

df = df.drop(drop_idx).reset_index(drop=True)
y = y.drop(drop_idx).reset_index(drop=True)

print(f"Cases eliminades: {len(drop_idx)}")
print(f"Shape resultant: {df.shape}")

      GrLivArea  SalePrice
523        4676   184750.0
1298       5642   160000.0
Cases eliminades: 2
Shape resultant: (2917, 237)


In [3]:
# Correlació de cada component amb SalePrice
components = ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']
for col in components:
    corr = df.iloc[:len(y)][col].corr(y)
    print(f"{col}: {corr:.3f}")

TotalBsmtSF: 0.648
1stFlrSF: 0.621
2ndFlrSF: 0.320


In [4]:
print(f"Cases amb 2n pis: {(df['2ndFlrSF'] > 0).sum()}")
print(f"Total cases: {len(df)}")
print(f"Percentatge: {(df['2ndFlrSF'] > 0).mean()*100:.1f}%")

Cases amb 2n pis: 1249
Total cases: 2917
Percentatge: 42.8%


In [5]:
# Superfície base — alta correlació confirmada
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF']

# 2n pis per separat
df['Has2ndFloor'] = (df['2ndFlrSF'] > 0).astype(int)

# Verifiquem correlació del TotalSF nou
corr_total = df.iloc[:len(y)]['TotalSF'].corr(y)
print(f"TotalSF correlació: {corr_total:.3f}")

TotalSF correlació: 0.668


C:\Users\User\AppData\Local\Temp\ipykernel_4328\2014626447.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF']
C:\Users\User\AppData\Local\Temp\ipykernel_4328\2014626447.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Has2ndFloor'] = (df['2ndFlrSF'] > 0).astype(int)


In [6]:
# Correlació dels banys per separat primer
baths = ['FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath']
for col in baths:
    corr = df.iloc[:len(y)][col].corr(y)
    print(f"{col}: {corr:.3f}")

FullBath: 0.596
HalfBath: 0.314
BsmtFullBath: 0.237
BsmtHalfBath: -0.005


In [7]:
df['TotalBaths'] = (df['FullBath'] + 
                    df['BsmtFullBath'] * 0.5 +
                    df['HalfBath'] * 0.3 +
                    df['BsmtHalfBath'] * 0)  # ignorem BsmtHalfBath

corr = df.iloc[:len(y)]['TotalBaths'].corr(y)
print(f"TotalBaths correlació: {corr:.3f}")

TotalBaths correlació: 0.697


C:\Users\User\AppData\Local\Temp\ipykernel_4328\1213647115.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TotalBaths'] = (df['FullBath'] +


In [8]:
porchs = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'WoodDeckSF']
for col in porchs:
    corr = df.iloc[:len(y)][col].corr(y)
    print(f"{col}: {corr:.3f}")

OpenPorchSF: 0.325
EnclosedPorch: -0.149
3SsnPorch: 0.055
ScreenPorch: 0.121
WoodDeckSF: 0.334


In [9]:
df['TotalPorchSF'] = (df['WoodDeckSF'] + 
                      df['OpenPorchSF'] + 
                      df['ScreenPorch'] * 0.4 +
                      df['3SsnPorch'] * 0.1)
# Ignorem EnclosedPorch

corr = df.iloc[:len(y)]['TotalPorchSF'].corr(y)
print(f"TotalPorchSF correlació: {corr:.3f}")

TotalPorchSF correlació: 0.454


C:\Users\User\AppData\Local\Temp\ipykernel_4328\1793376704.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TotalPorchSF'] = (df['WoodDeckSF'] +


In [11]:
quals = ['OverallQual', 'OverallCond', 'ExterQual', 'KitchenQual', 'GarageQual']
for col in quals:
    corr = df.iloc[:len(y)][col].corr(y)
    print(f"{col}: {corr:.3f}")

OverallQual: 0.821
OverallCond: -0.037
ExterQual: 0.682
KitchenQual: 0.670
GarageQual: 0.363


In [12]:
    # Qualitat × Superfície — cases grans de qualitat valen exponencialment més
df['Qual_TotalSF'] = df['OverallQual'] * df['TotalSF']

# Qualitat × GrLivArea
df['Qual_GrLivArea'] = df['OverallQual'] * df['GrLivArea']

for col in ['Qual_TotalSF', 'Qual_GrLivArea']:
    corr = df.iloc[:len(y)][col].corr(y)
    print(f"{col}: {corr:.3f}")

Qual_TotalSF: 0.805
Qual_GrLivArea: 0.838


C:\Users\User\AppData\Local\Temp\ipykernel_4328\766118885.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Qual_TotalSF'] = df['OverallQual'] * df['TotalSF']
C:\Users\User\AppData\Local\Temp\ipykernel_4328\766118885.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Qual_GrLivArea'] = df['OverallQual'] * df['GrLivArea']


In [13]:
df['GrLivArea_TotalSF'] = df['GrLivArea'] * df['TotalSF']
corr = df.iloc[:len(y)]['GrLivArea_TotalSF'].corr(y)
print(f"GrLivArea_TotalSF: {corr:.3f}")

GrLivArea_TotalSF: 0.764


C:\Users\User\AppData\Local\Temp\ipykernel_4328\3401614824.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['GrLivArea_TotalSF'] = df['GrLivArea'] * df['TotalSF']


In [14]:
# Barri premium + qualitat alta — el patró que vam veure al EDA
df['Qual_NridgHt'] = df['OverallQual'] * df['Neighborhood_NridgHt']
df['Qual_StoneBr']  = df['OverallQual'] * df['Neighborhood_StoneBr']

for col in ['Qual_NridgHt', 'Qual_StoneBr']:
    corr = df.iloc[:len(y)][col].corr(y)
    print(f"{col}: {corr:.3f}")

Qual_NridgHt: 0.366
Qual_StoneBr: 0.190


C:\Users\User\AppData\Local\Temp\ipykernel_4328\4183934245.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Qual_NridgHt'] = df['OverallQual'] * df['Neighborhood_NridgHt']
C:\Users\User\AppData\Local\Temp\ipykernel_4328\4183934245.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Qual_StoneBr']  = df['OverallQual'] * df['Neighborhood_StoneBr']


In [15]:
# Venda sobre plànol + qualitat — el patró que vam veure a l'EDA
df['Qual_Partial'] = df['OverallQual'] * df['SaleCondition_Partial']

corr = df.iloc[:len(y)]['Qual_Partial'].corr(y)
print(f"Qual_Partial: {corr:.3f}")

Qual_Partial: 0.356


C:\Users\User\AppData\Local\Temp\ipykernel_4328\1283665386.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Qual_Partial'] = df['OverallQual'] * df['SaleCondition_Partial']


In [16]:
df = df.copy()
print("DataFrame desfragmentat!")
print(f"Shape final: {df.shape}")

DataFrame desfragmentat!
Shape final: (2917, 247)


In [17]:
new_features = ['TotalSF', 'Has2ndFloor', 'TotalBaths', 'TotalPorchSF', 
                'Qual_GrLivArea', 'Qual_TotalSF', 'Qual_NridgHt', 
                'Qual_Partial', 'GrLivArea_TotalSF']

for col in new_features:
    corr = df.iloc[:len(y)][col].corr(y)
    print(f"{col}: {corr:.3f}")

TotalSF: 0.668
Has2ndFloor: 0.151
TotalBaths: 0.697
TotalPorchSF: 0.454
Qual_GrLivArea: 0.838
Qual_TotalSF: 0.805
Qual_NridgHt: 0.366
Qual_Partial: 0.356
GrLivArea_TotalSF: 0.764


In [18]:
df.drop('Has2ndFloor', axis=1, inplace=True)
print(f"Shape final: {df.shape}")

Shape final: (2917, 246)


In [19]:
# Guardem el dataset final
df.to_csv('../data/processed/df_features.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)
print("Dataset guardat!")

Dataset guardat!
